In [14]:
!pip install -q -U trl torchao bert-score git+https://github.com/bckim92/language-evaluation.git

from google.colab import drive
drive.mount("/content/drive")

import os, gc, re, json, math, random, zipfile, tarfile, urllib.request, subprocess
from pathlib import Path
import gdown
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from tqdm.auto import tqdm
from datasets import Dataset, Image as HFImage
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from peft import LoraConfig, PeftModel
from trl import DPOConfig, DPOTrainer
from trl.trainer.dpo_trainer import DataCollatorForVisionPreference

from bert_score import score as bertscore_score
from language_evaluation.coco_caption_py3.pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from language_evaluation.coco_caption_py3.pycocoevalcap.bleu.bleu import Bleu
from language_evaluation.coco_caption_py3.pycocoevalcap.rouge.rouge import Rouge
from language_evaluation.coco_caption_py3.pycocoevalcap.cider.cider import Cider
import language_evaluation.coco_caption_py3.pycocoevalcap.meteor.meteor as meteor_mod

# 1. Reproducibility & Hyperparameters (Exact match to E3 winner)
SEED = 10623
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
NUM_EPOCHS = 1
MICRO_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 5e-6
DPO_BETA = 0.05

# 2. Directories
PROJECT_DIR = Path("/content/drive/MyDrive/EMMbedding")
DATA_DIR = Path("/content/data/artcap")
EXP_NAME = "E2_random_dpo"
RUN_TAG = f"{EXP_NAME}_lr{LEARNING_RATE}_beta{DPO_BETA}_ep{NUM_EPOCHS}"

EXP_DIR = PROJECT_DIR / EXP_NAME / RUN_TAG
ADAPTER_DIR = EXP_DIR / "adapter"
PREDICTIONS_DIR = EXP_DIR / "predictions"
for d in [DATA_DIR, EXP_DIR, ADAPTER_DIR, PREDICTIONS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)} | Run: {RUN_TAG}")

# 3. Ensure Images Exist Locally & Remap Paths Safely
img_dir = DATA_DIR / "image_archive"
if not img_dir.exists() or not any(img_dir.iterdir()):
    archive = DATA_DIR / "artcap_images_archive"
    if not archive.exists():
        gdown.download("https://drive.google.com/uc?id=1inQ4W-CtNhhfo1vAehSbLgyMPTmltWVa", str(archive), quiet=False, fuzzy=True)
    img_dir.mkdir(parents=True, exist_ok=True)
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as z: z.extractall(img_dir)
    else:
        with tarfile.open(archive) as t: t.extractall(img_dir)

image_lookup = {p.stem: str(p) for p in DATA_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}

def resolve_img(image_id):
    s = str(image_id)
    return image_lookup.get(s) or image_lookup[str(int(s))]

# 4. Load Existing Canonical Splits & Saved Random Pairs from Drive
all_rows = [json.loads(line) for line in open(PROJECT_DIR / "captions.jsonl") if line.strip()]
for r in all_rows:
    r["image_path"] = resolve_img(r["image_id"])

test_rows = [r for r in all_rows if r["split"] == "test"]

PROMPT = """Describe the visual content of this painting in one concise, informative sentence.

Focus on:
- The people, objects, and subjects depicted.
- Their actions and spatial relationships.
- Important colors, visual attributes, and setting.

Only describe details that are visually supported by the image.
Do not speculate about the artist, historical context, or symbolic meaning.

Return only the caption."""

raw_pairs = [json.loads(line) for line in open(PROJECT_DIR / "preferences" / "random_pairs.jsonl") if line.strip()]
train_pairs = [r for r in raw_pairs if r["split"] == "train" and r["chosen"].strip() != r["rejected"].strip()]

train_dataset = Dataset.from_list([
    {
        "image": {"path": resolve_img(r["image_id"]), "bytes": None},
        "prompt": [{"role": "user", "content": PROMPT}],
        "chosen": [{"role": "assistant", "content": r["chosen"]}],
        "rejected": [{"role": "assistant", "content": r["rejected"]}],
        "image_id": str(r["image_id"]),
    }
    for r in train_pairs
]).cast_column("image", HFImage())

print(f"Loaded E2 Train Pairs: {len(train_dataset)} | KALE Test Paintings: {len(test_rows)}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-40GB | Run: E2_random_dpo_lr5e-06_beta0.05_ep1


Downloading...
From (original): https://drive.google.com/uc?id=1inQ4W-CtNhhfo1vAehSbLgyMPTmltWVa
From (redirected): https://drive.google.com/uc?id=1inQ4W-CtNhhfo1vAehSbLgyMPTmltWVa&confirm=t&uuid=3d3a88b7-a08e-484a-ad5d-e2eda7548683
To: /content/data/artcap/artcap_images_archive
100%|██████████| 597M/597M [00:06<00:00, 90.7MB/s]


Loaded E2 Train Pairs: 4532 | KALE Test Paintings: 1082


In [15]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    size={"shortest_edge": 224 * 28 * 28, "longest_edge": 512 * 28 * 28},
)
processor.tokenizer.padding_side = "left"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False

for name, param in model.named_parameters():
    if "visual" in name.lower():
        param.requires_grad = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=r".*language_model.*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
)

vision_collator = DataCollatorForVisionPreference(processor=processor, max_length=None)
effective_bs = MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
est_steps = math.ceil(len(train_dataset) / effective_bs) * NUM_EPOCHS

training_args = DPOConfig(
    output_dir=str(EXP_DIR / "checkpoints"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=MICRO_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    beta=DPO_BETA,
    loss_type="sigmoid",
    lr_scheduler_type="cosine",
    warmup_steps=int(0.05 * est_steps),
    optim="adamw_torch",
    max_grad_norm=0.3,
    bf16=True,
    tf32=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=None,
    logging_steps=10,
    save_strategy="no",
    remove_unused_columns=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=processor,
    peft_config=lora_config,
    data_collator=vision_collator,
)

torch.cuda.empty_cache()
train_result = trainer.train()

trainer.save_model(str(ADAPTER_DIR))
processor.save_pretrained(str(ADAPTER_DIR))

with open(EXP_DIR / "training_metrics.json", "w") as f:
    json.dump(train_result.metrics, f, indent=2, default=str)

del trainer, model
gc.collect()
torch.cuda.empty_cache()
print("Saved E2 (beta=0.05) adapter to:", ADAPTER_DIR)

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Step,Training Loss
10,0.693041
20,0.692904
30,0.692993
40,0.692915
50,0.694711
60,0.689930
70,0.690473
80,0.695123
90,0.699893
100,0.688421


Saved E2 (beta=0.05) adapter to: /content/drive/MyDrive/EMMbedding/E2_random_dpo/E2_random_dpo_lr5e-06_beta0.05_ep1/adapter


In [16]:
pred_csv = PREDICTIONS_DIR / "test_predictions.csv"
predictions = pd.read_csv(pred_csv).to_dict("records") if pred_csv.exists() else []
done_ids = {str(r["image_id"]) for r in predictions}
remaining = [r for r in test_rows if str(r["image_id"]) not in done_ids]

if remaining:
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True
    )
    inf_model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR)).to("cuda").eval()
    inf_model.config.use_cache = True

    BATCH_SIZE = 16
    for i in tqdm(range(0, len(remaining), BATCH_SIZE), desc="Generating (test)"):
        batch = remaining[i : i + BATCH_SIZE]
        msgs = [
            [{"role": "user", "content": [
                {"type": "image", "image": Image.open(r["image_path"]).convert("RGB")},
                {"type": "text", "text": PROMPT},
            ]}]
            for r in batch
        ]
        inputs = processor.apply_chat_template(
            msgs, tokenize=True, add_generation_prompt=True, padding=True, return_dict=True, return_tensors="pt"
        ).to("cuda")

        with torch.inference_mode():
            out_ids = inf_model.generate(**inputs, max_new_tokens=128, do_sample=False)

        trimmed = [o[len(inp):] for inp, o in zip(inputs["input_ids"], out_ids)]
        captions = [
            t.strip()
            for t in processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        ]
        for r, cap in zip(batch, captions):
            predictions.append({
                "image_id": str(r["image_id"]),
                "prediction": cap,
                "references": json.dumps(r["references"]),
            })
        pd.DataFrame(predictions).to_csv(pred_csv, index=False)

    del inf_model, base_model
    gc.collect()
    torch.cuda.empty_cache()

print(f"Saved {len(predictions)} test predictions to {pred_csv}")

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Generating (test):   0%|          | 0/68 [00:00<?, ?it/s]

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

Saved 1082 test predictions to /content/drive/MyDrive/EMMbedding/E2_random_dpo/E2_random_dpo_lr5e-06_beta0.05_ep1/predictions/test_predictions.csv


In [17]:
meteor_dir = os.path.dirname(meteor_mod.__file__)
data_dir = os.path.join(meteor_dir, "data")
os.makedirs(data_dir, exist_ok=True)
paraphrase_path = os.path.join(data_dir, "paraphrase-en.gz")
if not os.path.exists(paraphrase_path) or os.path.getsize(paraphrase_path) == 0:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/tylin/coco-caption/master/pycocoevalcap/meteor/data/paraphrase-en.gz",
        paraphrase_path,
    )

def compute_meteor_clean(gts, res):
    jar_path = os.path.join(meteor_dir, "meteor-1.5.jar")
    proc = subprocess.Popen(
        ["java", "-Xmx2G", "-XX:-UseContainerSupport", "-Xlog:all=off", "-jar", jar_path, "-", "-", "-stdio", "-l", "en", "-norm"],
        cwd=meteor_dir, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
    )
    imgIds = list(gts.keys())
    eval_line = "EVAL"
    for i in imgIds:
        hyp = res[i][0].replace("|||", "").replace("  ", " ")
        refs_str = " ||| ".join(gts[i])
        proc.stdin.write(f"SCORE ||| {refs_str} ||| {hyp}\n".encode("utf-8"))
        proc.stdin.flush()
        stat = proc.stdout.readline().decode("utf-8").strip()
        eval_line += f" ||| {stat}"
    proc.stdin.write(f"{eval_line}\n".encode("utf-8"))
    proc.stdin.flush()
    _ = [proc.stdout.readline().decode("utf-8").strip() for _ in imgIds]
    agg_line = proc.stdout.readline().decode("utf-8").strip()
    if " " in agg_line:
        proc.stdin.write(f"SCORE ||| {agg_line}\n".encode("utf-8"))
        proc.stdin.flush()
        score = float(proc.stdout.readline().decode("utf-8").strip())
    else:
        score = float(agg_line)
    proc.kill()
    return score

df = pd.read_csv(PREDICTIONS_DIR / "test_predictions.csv")
preds = df["prediction"].astype(str).tolist()
refs = [json.loads(x) for x in df["references"]]

tokenizer = PTBTokenizer()
gts = tokenizer.tokenize({i: [{"caption": c} for c in r] for i, r in enumerate(refs)})
res = tokenizer.tokenize({i: [{"caption": p}] for i, p in enumerate(preds)})

cider_s, _ = Cider().compute_score(gts, res)
bleu_s, _ = Bleu(4).compute_score(gts, res)
rouge_s, _ = Rouge().compute_score(gts, res)
meteor_s = compute_meteor_clean(gts, res)
bert_P, bert_R, bert_F1 = bertscore_score(
    cands=preds, refs=refs, lang="en", model_type="roberta-large",
    device="cuda", batch_size=16, verbose=False, rescale_with_baseline=False,
)

results = {
    "CIDEr": float(cider_s),
    "avg_pred_words": float(np.mean([len(p.split()) for p in preds])),
    "Bleu_1": float(bleu_s[0]),
    "Bleu_2": float(bleu_s[1]),
    "Bleu_3": float(bleu_s[2]),
    "Bleu_4": float(bleu_s[3]),
    "ROUGE_L": float(rouge_s),
    "BERTScore_P": float(bert_P.mean().item()),
    "BERTScore_R": float(bert_R.mean().item()),
    "BERTScore_F1": float(bert_F1.mean().item()),
    "METEOR": float(meteor_s),
}

with open(EXP_DIR / "test_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

pct_metrics = {"Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4", "METEOR", "ROUGE_L", "CIDEr", "BERTScore_P", "BERTScore_R", "BERTScore_F1"}
final_df = pd.DataFrame([
    {"Metric": k, "Raw Score": v, "Reported Score": v * 100 if k in pct_metrics else v}
    for k, v in results.items()
])
final_df.to_csv(EXP_DIR / "final_e2_beta005_test_metrics.csv", index=False)
display(final_df.style.format({"Raw Score": "{:.4f}", "Reported Score": "{:.4f}"}))

{'testlen': 40389, 'reflen': 16877, 'guess': [40389, 39307, 38225, 37143], 'correct': [8443, 1643, 216, 65]}
ratio: 2.3931385909816676


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,Metric,Raw Score,Reported Score
0,CIDEr,0.0038,0.3831
1,avg_pred_words,37.2495,37.2495
2,Bleu_1,0.2090,20.9042
3,Bleu_2,0.0935,9.3476
4,Bleu_3,0.0367,3.6686
5,Bleu_4,0.0171,1.7145
6,ROUGE_L,0.1990,19.9017
7,BERTScore_P,0.8526,85.2640
8,BERTScore_R,0.9114,91.1384
9,BERTScore_F1,0.8788,87.8841


In [ ]:
# ============================================================
# APPENDIX / REPRODUCIBILITY: RANDOM PREFERENCE PAIR GENERATION
# ============================================================
from collections import Counter
from itertools import combinations

PREFERENCES_DIR = PROJECT_DIR / "preferences"
PREFERENCES_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_PAIRS_PATH = PREFERENCES_DIR / "random_pairs.jsonl"
JUDGE_PAIRS_PATH = PROJECT_DIR / "judge_pairs.jsonl"

if RANDOM_PAIRS_PATH.exists():
    repro_pairs = [json.loads(line) for line in open(RANDOM_PAIRS_PATH) if line.strip()]
    print(f"Verified existing {RANDOM_PAIRS_PATH} ({len(repro_pairs)} pairs).")
else:
    judge_pairs = [json.loads(line) for line in open(JUDGE_PAIRS_PATH) if line.strip()]

    # Match exact pair counts per painting & split from judge_pairs.jsonl
    # Uses ONLY the counts, never the judge's caption preferences.
    target_pair_counts = Counter(
        (str(row["image_id"]), row["split"])
        for row in judge_pairs
    )

    artcap_lookup = {
        (str(row["image_id"]), row["split"]): row
        for row in all_rows
        if row["split"] in {"train", "val"}
    }

    random_rng = random.Random(SEED)
    random_pairs = []

    for (image_id, split), requested_count in sorted(target_pair_counts.items()):
        key = (image_id, split)
        references = artcap_lookup[key]["references"]

        possible_pairs = list(combinations(range(len(references)), 2))
        valid_pairs = [
            (i, j)
            for i, j in possible_pairs
            if references[i].strip() != references[j].strip()
        ]

        if len(valid_pairs) < requested_count:
            raise ValueError(f"Insufficient distinct pairs for {key}: {len(valid_pairs)} < {requested_count}")

        sampled_pairs = random_rng.sample(valid_pairs, k=requested_count)

        for pair_number, (i, j) in enumerate(sampled_pairs, start=1):
            caption_a, caption_b = references[i], references[j]
            if random_rng.random() < 0.5:
                chosen, rejected = caption_a, caption_b
            else:
                chosen, rejected = caption_b, caption_a

            random_pairs.append({
                "pair_id": f"{image_id}:random:{pair_number}",
                "image_id": image_id,
                "split": split,
                "image_path": artcap_lookup[key]["image_path"],
                "chosen": chosen,
                "rejected": rejected,
            })

    assert all(r["chosen"].strip() != r["rejected"].strip() for r in random_pairs)
    assert Counter((str(r["image_id"]), r["split"]) for r in random_pairs) == target_pair_counts

    with open(RANDOM_PAIRS_PATH, "w") as f:
        for row in random_pairs:
            f.write(json.dumps(row) + "\n")

    print(f"Generated and saved {len(random_pairs)} random pairs to {RANDOM_PAIRS_PATH}")